# LOSO Cross-Validation Results Visualization
Fold별 best model로 validation slide에 대한 prediction 시각화

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from pathlib import Path
from tqdm import tqdm
import openslide
import cv2
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report, f1_score
from torch.utils.data import DataLoader

import sys
sys.path.insert(0, '/app/Gland_Seg/Code')
from config import Config
from dataset import PatchDataset, get_val_transforms
from model import create_model

config = Config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Load all fold checkpoints and run inference
slide_names = list(config.slides.keys())
fold_results = {}

for fold_idx, val_slide in enumerate(slide_names):
    fold_num = fold_idx + 1
    ckpt_path = Path(config.checkpoint_dir) / f'best_model_fold{fold_num}.pth'
    if not ckpt_path.exists():
        print(f'Fold {fold_num}: checkpoint not found, skipping')
        continue

    # Load model
    ckpt = torch.load(ckpt_path, weights_only=False, map_location=device)
    model = create_model(num_classes=config.num_classes, pretrained=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model = model.to(device)
    model.eval()

    # Val dataset
    val_dataset = PatchDataset(
        config.output_dir, [val_slide], config.slides,
        transform=get_val_transforms(config.input_size),
    )
    val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=4, pin_memory=True)

    # Inference
    all_preds = []
    all_labels = []
    all_probs = []
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f'Fold {fold_num} inference', leave=False):
            images = images.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            _, preds = outputs.max(1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)

    # Get patch coordinates from filenames
    coords = []
    for path, _ in val_dataset.samples:
        parts = path.stem.split('_')
        x, y = int(parts[-2]), int(parts[-1])
        coords.append((x, y))
    coords = np.array(coords)

    acc = (all_preds == all_labels).mean()
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    val_class = config.slides[val_slide]['class']

    fold_results[fold_num] = {
        'val_slide': val_slide,
        'val_class': val_class,
        'epoch': ckpt['epoch'],
        'preds': all_preds,
        'labels': all_labels,
        'probs': all_probs,
        'coords': coords,
        'acc': acc,
        'f1': f1,
    }
    print(f'Fold {fold_num}: val={val_slide} ({val_class}), epoch={ckpt["epoch"]}, Acc={acc:.4f}, F1={f1:.4f}')

In [ ]:
# Summary table
print(f"{'Fold':<6} {'Val Slide':<20} {'Class':<12} {'Epoch':>6} {'Acc':>8} {'F1':>8}")
print('-' * 64)
accs, f1s = [], []
for fold_num, r in fold_results.items():
    print(f"{fold_num:<6} {r['val_slide']:<20} {r['val_class']:<12} {r['epoch']:>6} {r['acc']:>8.4f} {r['f1']:>8.4f}")
    accs.append(r['acc'])
    f1s.append(r['f1'])
print('-' * 64)
print(f"{'Mean':<40} {np.mean(accs):>8.4f} {np.mean(f1s):>8.4f}")
print(f"{'Std':<40} {np.std(accs):>8.4f} {np.std(f1s):>8.4f}")

In [ ]:
# Confusion matrices for all folds
fig, axes = plt.subplots(1, 5, figsize=(25, 5))

for ax, (fold_num, r) in zip(axes, fold_results.items()):
    cm = confusion_matrix(r['labels'], r['preds'], labels=[0, 1])
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(config.class_names, fontsize=10)
    ax.set_yticklabels(config.class_names, fontsize=10)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=13)
    ax.set_title(f"Fold {fold_num}: {r['val_slide'].split('-', 1)[1]}\n"
                 f"({r['val_class']}) Acc={r['acc']:.3f} F1={r['f1']:.3f}", fontsize=11)

plt.suptitle('Confusion Matrices - LOSO Cross-Validation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/app/Gland_Seg/Viz/cv_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Prediction heatmap overlay on slide thumbnails
fig, axes = plt.subplots(1, 5, figsize=(30, 8))

for ax, (fold_num, r) in zip(axes, fold_results.items()):
    slide_name = r['val_slide']
    info = config.slides[slide_name]
    svs_path = Path(config.svs_dir) / info['svs']
    slide = openslide.OpenSlide(str(svs_path))
    viz_level = min(3, slide.level_count - 1)
    ds = slide.level_downsamples[viz_level]
    thumb = slide.read_region((0, 0), viz_level, slide.level_dimensions[viz_level]).convert('RGB')
    thumb_np = np.array(thumb)
    ax.imshow(thumb_np)

    ps = config.patch_size / ds
    coords = r['coords']
    preds = r['preds']
    labels = r['labels']

    for idx in range(len(preds)):
        x, y = coords[idx]
        correct = preds[idx] == labels[idx]
        if correct:
            color = 'green'
            alpha = 0.2
        else:
            color = 'red'
            alpha = 0.5
        rect = plt.Rectangle((x/ds, y/ds), ps, ps,
                           linewidth=0.3, edgecolor=color, facecolor=color, alpha=alpha)
        ax.add_patch(rect)

    n_correct = (preds == labels).sum()
    n_wrong = (preds != labels).sum()
    ax.set_title(f"Fold {fold_num}: {slide_name}\n"
                 f"GT={r['val_class']} | correct={n_correct} (green) wrong={n_wrong} (red)", fontsize=10)
    ax.axis('off')
    slide.close()

legend_elements = [
    Patch(facecolor='green', alpha=0.4, label='Correct'),
    Patch(facecolor='red', alpha=0.6, label='Wrong'),
]
axes[-1].legend(handles=legend_elements, loc='lower right', fontsize=10)

plt.suptitle('Prediction Results on Validation Slides (LOSO-CV)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/app/Gland_Seg/Viz/cv_prediction_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Sample wrong predictions per fold
for fold_num, r in fold_results.items():
    wrong_mask = r['preds'] != r['labels']
    n_wrong = wrong_mask.sum()
    if n_wrong == 0:
        print(f'Fold {fold_num}: No wrong predictions')
        continue

    val_slide = r['val_slide']
    val_dataset = PatchDataset(
        config.output_dir, [val_slide], config.slides, transform=None,
    )

    wrong_indices = np.where(wrong_mask)[0]
    n_show = min(10, len(wrong_indices))
    sample_indices = wrong_indices[:n_show]

    fig, axes = plt.subplots(1, n_show, figsize=(3 * n_show, 3))
    if n_show == 1:
        axes = [axes]
    for ax_idx, idx in enumerate(sample_indices):
        img_path, label = val_dataset.samples[idx]
        img = Image.open(img_path).resize((224, 224))
        axes[ax_idx].imshow(img)
        pred_name = config.class_names[r['preds'][idx]]
        true_name = config.class_names[r['labels'][idx]]
        prob = r['probs'][idx][r['preds'][idx]]
        axes[ax_idx].set_title(f'True: {true_name}\nPred: {pred_name}\np={prob:.2f}', fontsize=9)
        axes[ax_idx].axis('off')

    plt.suptitle(f'Fold {fold_num} Wrong Predictions ({n_wrong} total) - {val_slide}', fontsize=12)
    plt.tight_layout()
    plt.savefig(f'/app/Gland_Seg/Viz/cv_wrong_fold{fold_num}.png', dpi=150, bbox_inches='tight')
    plt.show()